In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

Mounted at /content/drive


In [3]:
files = ['branches','customers','inventory_master','invoices','payments',
         'products','purchase_orders_header','purchase_orders_lines',
         'sales_orders_header','sales_orders_lines','stock_ledger','suppliers']

results = []
for name in files:
    df = pd.read_csv(BASE_PATH + name + '.csv')
    for col in df.columns:
        if 'id' in col.lower():
            dup_rows = df[col].duplicated(keep=False).sum()
            results.append({
                'table': name,
                'column': col,
                'total_rows': len(df),
                'unique_values': df[col].nunique(),
                'nulls': df[col].isna().sum(),
                'rows_with_repeated_id': dup_rows
            })

report = pd.DataFrame(results)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 200)
print(report)

                     table        column  total_rows  unique_values  nulls  rows_with_repeated_id
0                 branches     branch_id           6              6      0                      0
1                 branches    manager_id           6              6      0                      0
2                customers   customer_id         500            500      0                      0
3                customers     branch_id         500              6      0                    500
4         inventory_master    product_id         180             30      0                    180
5         inventory_master     branch_id         180              6      0                    180
6                 invoices    invoice_id       18033          17836      0                    393
7                 invoices         so_id       18033          18033      0                      0
8                 invoices   customer_id       18033            500      0                  18033
9                 in

In [4]:
names = ['branches','customers','inventory_master','invoices','payments',
         'products','purchase_orders_header','purchase_orders_lines',
         'sales_orders_header','sales_orders_lines','stock_ledger','suppliers']

tables = {}
for n in names:
    tables[n] = pd.read_csv(BASE_PATH + n + '.csv')

# stock_ledger.reference_id can point to a purchase order or a sales order
tables['po_or_so'] = pd.DataFrame({
    'id': pd.concat([tables['purchase_orders_header']['po_id'],
                     tables['sales_orders_header']['so_id']])
})

links = [
    ('customers',              'branch_id',   'branches',               'branch_id'),
    ('inventory_master',       'product_id',  'products',               'product_id'),
    ('inventory_master',       'branch_id',   'branches',               'branch_id'),
    ('invoices',               'customer_id', 'customers',              'customer_id'),
    ('invoices',               'branch_id',   'branches',               'branch_id'),
    ('invoices',               'so_id',       'sales_orders_header',    'so_id'),
    ('payments',               'invoice_id',  'invoices',               'invoice_id'),
    ('purchase_orders_header', 'supplier_id', 'suppliers',              'supplier_id'),
    ('purchase_orders_header', 'branch_id',   'branches',               'branch_id'),
    ('purchase_orders_lines',  'po_id',       'purchase_orders_header', 'po_id'),
    ('purchase_orders_lines',  'product_id',  'products',               'product_id'),
    ('sales_orders_header',    'customer_id', 'customers',              'customer_id'),
    ('sales_orders_header',    'branch_id',   'branches',               'branch_id'),
    ('sales_orders_lines',     'so_id',       'sales_orders_header',    'so_id'),
    ('sales_orders_lines',     'product_id',  'products',               'product_id'),
    ('stock_ledger',           'product_id',  'products',               'product_id'),
    ('stock_ledger',           'branch_id',   'branches',               'branch_id'),
    ('stock_ledger',           'reference_id','po_or_so',               'id'),
]

rows = []
for child, ccol, parent, pcol in links:
    parent_ids = set(tables[parent][pcol])
    orphans = (~tables[child][ccol].isin(parent_ids)).sum()
    total = len(tables[child])
    rows.append({
        'child_table': child,
        'column': ccol,
        'points_to': parent + '.' + pcol,
        'total_rows': total,
        'orphan_rows': orphans,
        'orphan_pct': round(orphans / total * 100, 2)
    })

orphan_report = pd.DataFrame(rows)
pd.set_option('display.width', 200)
print(orphan_report)

               child_table        column                     points_to  total_rows  orphan_rows  orphan_pct
0                customers     branch_id            branches.branch_id         500            0        0.00
1         inventory_master    product_id           products.product_id         180            0        0.00
2         inventory_master     branch_id            branches.branch_id         180            0        0.00
3                 invoices   customer_id         customers.customer_id       18033            0        0.00
4                 invoices     branch_id            branches.branch_id       18033            0        0.00
5                 invoices         so_id     sales_orders_header.so_id       18033            0        0.00
6                 payments    invoice_id           invoices.invoice_id       19257            0        0.00
7   purchase_orders_header   supplier_id         suppliers.supplier_id       24000            0        0.00
8   purchase_orders_header  

In [5]:
ledger = tables['stock_ledger']
valid_ids = set(tables['po_or_so']['id'])
orphan_rows = ledger[~ledger['reference_id'].isin(valid_ids)]

print(orphan_rows.head())
print()
for col in orphan_rows.columns:
    if orphan_rows[col].dtype == 'object' and orphan_rows[col].nunique() <= 10:
        print(orphan_rows[col].value_counts())
        print()

      movement_id product_id branch_id movement_type movement_date  quantity reference_type reference_id  running_balance
48   MOV-ADJ-1519       P001    AHM001    ADJUSTMENT    2019-04-14        38            ADJ     ADJ-1519             4207
131  MOV-ADJ-1097       P001    AHM001    ADJUSTMENT    2019-08-29        38            ADJ     ADJ-1097            10990
171   MOV-ADJ-524       P001    AHM001    ADJUSTMENT    2019-11-01        33            ADJ      ADJ-524            13979
419   MOV-ADJ-920       P001    AHM001    ADJUSTMENT    2020-11-21        20            ADJ      ADJ-920            37801
518  MOV-ADJ-1040       P001    AHM001    ADJUSTMENT    2021-05-14        50            ADJ     ADJ-1040            46348

branch_id
CHN001    429
KOL001    424
HYD001    423
PUN001    400
DEL001    392
AHM001    386
Name: count, dtype: int64

movement_type
ADJUSTMENT    2454
Name: count, dtype: int64

reference_type
ADJ    2454
Name: count, dtype: int64



In [6]:
rows = []
for n in names:
    df = tables[n]
    for col in df.columns:
        if col.endswith('_id'):
            continue
        s = df[col]
        row = {'table': n, 'column': col, 'dtype': str(s.dtype),
               'nulls': int(s.isna().sum()), 'unique': s.nunique()}
        if pd.api.types.is_numeric_dtype(s):
            row['min'] = s.min()
            row['max'] = s.max()
            row['negatives'] = int((s < 0).sum())
            row['zeros'] = int((s == 0).sum())
        rows.append(row)

profile = pd.DataFrame(rows)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 250)
print(profile.to_string())

                      table                    column    dtype  nulls  unique           min           max  negatives  zeros
0                  branches               branch_name   object      0       6           NaN           NaN        NaN    NaN
1                  branches                      city   object      0       6           NaN           NaN        NaN    NaN
2                  branches                     state   object      0       6           NaN           NaN        NaN    NaN
3                  branches                    region   object      0       4           NaN           NaN        NaN    NaN
4                  branches            warehouse_type   object      0       4           NaN           NaN        NaN    NaN
5                  branches        warehouse_capacity   object      0       6           NaN           NaN        NaN    NaN
6                  branches  service_center_available   object      0       2           NaN           NaN        NaN    NaN
7       

In [7]:
print(tables['branches'][['branch_id', 'warehouse_capacity']])
print()

po = tables['purchase_orders_header']
print(po['po_status'].value_counts())
print()
print(po[po['received_date'].isna()]['po_status'].value_counts())

  branch_id warehouse_capacity
0    DEL001         45230 sqft
1    PUN001         38750 sqft
2    CHN001         41890 sqft
3    HYD001         36210 sqft
4    KOL001         29880 sqft
5    AHM001         34120 sqft

po_status
Received     21630
Cancelled     2370
Name: count, dtype: int64

po_status
Cancelled    2370
Name: count, dtype: int64


In [8]:
po = tables['purchase_orders_header'].copy()
po = po[po['po_status'] == 'Received'].copy()

for c in ['order_date', 'expected_delivery_date', 'received_date']:
    po[c] = pd.to_datetime(po[c])

po['actual_days'] = (po['received_date'] - po['order_date']).dt.days
po['planned_days'] = (po['expected_delivery_date'] - po['order_date']).dt.days

by_supplier = po.groupby('supplier_id').agg(
    actual_avg=('actual_days', 'mean'),
    planned_avg=('planned_days', 'mean')
).round(1)
by_supplier['supplier_table_days'] = tables['suppliers'].set_index('supplier_id')['lead_time_days']
print(by_supplier)
print()
print(po[['actual_days', 'planned_days']].describe().round(1))
print()

lines = tables['purchase_orders_lines'][['po_id', 'product_id']].drop_duplicates()
lp = lines.merge(po[['po_id', 'actual_days']], on='po_id')
by_product = lp.groupby('product_id')['actual_days'].mean().round(1).to_frame('actual_avg')
by_product['product_table_days'] = tables['products'].set_index('product_id')['lead_time_days']
print(by_product)

             actual_avg  planned_avg  supplier_table_days
supplier_id                                              
SUP0001            19.5         19.0                   19
SUP0002            10.5         10.0                   10
SUP0003            16.5         16.0                   16
SUP0004            18.5         18.0                   18
SUP0005            28.5         28.0                   28
SUP0006            10.5         10.0                   10
SUP0007            12.5         12.0                   12
SUP0008            26.5         26.0                   26

       actual_days  planned_days
count      21630.0       21630.0
mean          17.9          17.4
std            6.7           6.4
min            7.0          10.0
25%           12.0          12.0
50%           18.0          16.0
75%           25.0          26.0
max           35.0          28.0

            actual_avg  product_table_days
product_id                                
P001              17.7             

In [9]:
date_cols = {
    'invoices': ['invoice_date', 'due_date'],
    'payments': ['payment_date'],
    'purchase_orders_header': ['order_date', 'expected_delivery_date', 'received_date'],
    'sales_orders_header': ['order_date', 'delivery_date'],
    'stock_ledger': ['movement_date'],
    'customers': ['customer_since', 'last_purchase_date'],
    'products': ['last_purchase_date'],
}

rows = []
for t, cols in date_cols.items():
    for c in cols:
        raw = tables[t][c]
        converted = pd.to_datetime(raw, errors='coerce')
        rows.append({
            'table': t,
            'column': c,
            'failed_to_convert': int(converted.isna().sum() - raw.isna().sum()),
            'earliest': str(converted.min())[:10],
            'latest': str(converted.max())[:10],
        })

print(pd.DataFrame(rows))

                     table                  column  failed_to_convert    earliest      latest
0                 invoices            invoice_date                  0  2019-01-03  2025-01-13
1                 invoices                due_date                  0  2019-01-20  2025-03-14
2                 payments            payment_date                  0  2019-01-09  2025-03-20
3   purchase_orders_header              order_date                  0  2019-01-01  2024-12-31
4   purchase_orders_header  expected_delivery_date                  0  2019-01-11  2025-01-28
5   purchase_orders_header           received_date                  0  2019-01-11  2025-01-28
6      sales_orders_header              order_date                  0  2019-01-01  2024-12-31
7      sales_orders_header           delivery_date                  0  2019-01-02  2025-01-13
8             stock_ledger           movement_date                  0  2019-01-01  2025-01-28
9                customers          customer_since          

In [10]:
inv = tables['invoices'].copy()
for c in ['invoice_date', 'due_date']:
    inv[c] = pd.to_datetime(inv[c])
print('Invoices: due_date before invoice_date:', (inv['due_date'] < inv['invoice_date']).sum())

po = tables['purchase_orders_header'].copy()
for c in ['order_date', 'expected_delivery_date', 'received_date']:
    po[c] = pd.to_datetime(po[c])
print('POs: expected delivery before order date:', (po['expected_delivery_date'] < po['order_date']).sum())
print('POs: received before order date:', (po['received_date'] < po['order_date']).sum())

so = tables['sales_orders_header'].copy()
for c in ['order_date', 'delivery_date']:
    so[c] = pd.to_datetime(so[c])
print('Sales orders: delivery before order date:', (so['delivery_date'] < so['order_date']).sum())

m = inv.merge(so[['so_id', 'order_date', 'delivery_date']], on='so_id')
print('Invoices dated before their sales order:', (m['invoice_date'] < m['order_date']).sum())
print('Invoices dated before delivery:', (m['invoice_date'] < m['delivery_date']).sum())

cust = tables['customers'].copy()
for c in ['customer_since', 'last_purchase_date']:
    cust[c] = pd.to_datetime(cust[c])
act = so.groupby('customer_id')['order_date'].agg(first_order='min', last_order='max')
cust = cust.set_index('customer_id').join(act)
print('Customers: last_purchase_date before their real last order:', (cust['last_purchase_date'] < cust['last_order']).sum())
print('Customers: customer_since after their real first order:', (cust['customer_since'] > cust['first_order']).sum())

Invoices: due_date before invoice_date: 0
POs: expected delivery before order date: 0
POs: received before order date: 0
Sales orders: delivery before order date: 0
Invoices dated before their sales order: 0
Invoices dated before delivery: 0
Customers: last_purchase_date before their real last order: 467
Customers: customer_since after their real first order: 293


In [11]:
cats = {
    'invoices': ['payment_status'],
    'payments': ['payment_method'],
    'sales_orders_header': ['order_status', 'payment_terms', 'sales_channel'],
    'purchase_orders_header': ['po_status'],
    'stock_ledger': ['movement_type', 'reference_type'],
    'customers': ['customer_type', 'payment_terms', 'industry_segment'],
    'products': ['criticality_level', 'usage_frequency', 'uom'],
}
for t, cols in cats.items():
    for c in cols:
        print(t + '.' + c)
        print(tables[t][c].value_counts())
        print()

problems = []
for n in names:
    df = tables[n]
    for c in df.columns:
        if df[c].dtype == 'object':
            s = df[c].dropna().astype(str)
            spaces = (s != s.str.strip()).sum()
            variants = s.nunique() - s.str.strip().str.lower().nunique()
            if spaces or variants:
                problems.append((n, c, spaces, variants))
print('Text columns with spacing or capital-letter problems:', problems)

invoices.payment_status
payment_status
Paid              12536
Partially Paid     3660
Unpaid             1837
Name: count, dtype: int64

payments.payment_method
payment_method
UPI              3905
Credit Card      3887
Bank Transfer    3871
Cheque           3819
Cash             3775
Name: count, dtype: int64

sales_orders_header.order_status
order_status
Delivered    18033
Cancelled     1967
Name: count, dtype: int64

sales_orders_header.payment_terms
payment_terms
Net 45     4485
Net 60     4235
Advance    4183
Net 30     3604
Net 15     3493
Name: count, dtype: int64

sales_orders_header.sales_channel
sales_channel
Field Sales       5036
Dealer Network    5028
Counter Sale      5014
Online            4922
Name: count, dtype: int64

purchase_orders_header.po_status
po_status
Received     21630
Cancelled     2370
Name: count, dtype: int64

stock_ledger.movement_type
movement_type
IN            127611
OUT           107165
ADJUSTMENT      2454
Name: count, dtype: int64

stock_ledger.r

In [12]:
so = tables['sales_orders_header']
po = tables['purchase_orders_header']
led = tables['stock_ledger']

canc_so = set(so.loc[so['order_status'] == 'Cancelled', 'so_id'])
canc_po = set(po.loc[po['po_status'] == 'Cancelled', 'po_id'])
ledger_ids = set(led['reference_id'])

print('Cancelled sales orders:', len(canc_so))
print('  with stock movements (rows):', led['reference_id'].isin(canc_so).sum())
print('  with invoices (rows):', tables['invoices']['so_id'].isin(canc_so).sum())
print('  with order lines (rows):', tables['sales_orders_lines']['so_id'].isin(canc_so).sum())
print()
print('Cancelled purchase orders:', len(canc_po))
print('  with stock movements (rows):', led['reference_id'].isin(canc_po).sum())
print('  with order lines (rows):', tables['purchase_orders_lines']['po_id'].isin(canc_po).sum())
print()
print('Delivered sales orders with NO stock movement:',
      (~so.loc[so['order_status'] == 'Delivered', 'so_id'].isin(ledger_ids)).sum())
print('Received purchase orders with NO stock movement:',
      (~po.loc[po['po_status'] == 'Received', 'po_id'].isin(ledger_ids)).sum())

Cancelled sales orders: 1967
  with stock movements (rows): 1275
  with invoices (rows): 0
  with order lines (rows): 12746

Cancelled purchase orders: 2370
  with stock movements (rows): 1542
  with order lines (rows): 15418

Delivered sales orders with NO stock movement: 162
Received purchase orders with NO stock movement: 239


In [13]:
import numpy as np

sl = tables['sales_orders_lines']
sh = tables['sales_orders_header'].set_index('so_id')
print('Sales lines: line_total is not quantity x unit_price:',
      (sl['line_total'] != sl['quantity'] * sl['unit_price']).sum())
print('Sales lines: gst_amount is off:',
      (~np.isclose(sl['gst_amount'], sl['line_total'] * sl['gst_rate'] / 100, atol=1)).sum())
print('Sales lines: line_grand_total is off:',
      (~np.isclose(sl['line_grand_total'], sl['line_total'] + sl['gst_amount'], atol=1)).sum())

sums = sl.groupby('so_id')['line_total'].sum().reindex(sh.index)
print('Sales headers: total_order_value differs from sum of lines:',
      (sh['total_order_value'] != sums).sum())
print('Sales headers: grand_total is off:',
      (~np.isclose(sh['grand_total'], sh['total_order_value'] + sh['total_gst_amount'], atol=1)).sum())

iv = tables['invoices'].set_index('so_id')
hh = sh.loc[iv.index]
print('Invoices: total_order_value differs from sales header:',
      (iv['total_order_value'] != hh['total_order_value']).sum())
print('Invoices: grand_total differs from sales header:',
      (~np.isclose(iv['grand_total'], hh['grand_total'], atol=1)).sum())

pl = tables['purchase_orders_lines']
ph = tables['purchase_orders_header'].set_index('po_id')
print('PO lines: line_total is not quantity x unit_cost:',
      (~np.isclose(pl['line_total'], pl['quantity'] * pl['unit_cost'], atol=1)).sum())
po_sums = pl.groupby('po_id')['line_total'].sum().reindex(ph.index)
print('PO headers: total_cost differs from sum of lines:',
      (~np.isclose(ph['total_cost'], po_sums, atol=1)).sum())

pr = tables['products']
print('Products: margin_percentage not matching price-based formula:',
      (~np.isclose(pr['margin_percentage'], (pr['unit_price'] - pr['unit_cost']) / pr['unit_price'] * 100, atol=0.5)).sum(),
      '| cost-based formula:',
      (~np.isclose(pr['margin_percentage'], (pr['unit_price'] - pr['unit_cost']) / pr['unit_cost'] * 100, atol=0.5)).sum())

cust = tables['customers'].set_index('customer_id')
real = tables['sales_orders_header'][tables['sales_orders_header']['order_status'] == 'Delivered'] \
    .groupby('customer_id')['total_order_value'].sum().reindex(cust.index)
print('Customers: total_purchase_value differs from real delivered orders by more than 1%:',
      ((cust['total_purchase_value'] - real).abs() > 0.01 * real).sum())

Sales lines: line_total is not quantity x unit_price: 0
Sales lines: gst_amount is off: 0
Sales lines: line_grand_total is off: 0
Sales headers: total_order_value differs from sum of lines: 0
Sales headers: grand_total is off: 0
Invoices: total_order_value differs from sales header: 0
Invoices: grand_total differs from sales header: 0
PO lines: line_total is not quantity x unit_cost: 0
PO headers: total_cost differs from sum of lines: 0
Products: margin_percentage not matching price-based formula: 29 | cost-based formula: 1
Customers: total_purchase_value differs from real delivered orders by more than 1%: 500


In [14]:
import numpy as np

led = tables['stock_ledger'].copy()
grp = led.groupby(['product_id', 'branch_id'])

led['prev_balance'] = grp['running_balance'].shift(1)
led['change'] = led['running_balance'] - led['prev_balance']
led['cancelled_order'] = led['reference_id'].isin(canc_so | canc_po)

chk = led.dropna(subset=['prev_balance']).copy()
chk['result'] = np.where(chk['change'] == chk['quantity'], 'adds quantity',
                np.where(chk['change'] == -chk['quantity'], 'subtracts quantity', 'other'))

print(pd.crosstab([chk['movement_type'], chk['cancelled_order']], chk['result']))
print()
print('Rows dated earlier than the row before them (same product and branch):',
      (grp['movement_date'].shift(1) > led['movement_date']).sum())

result                         adds quantity  subtracts quantity
movement_type cancelled_order                                   
ADJUSTMENT    False                     1219                1222
IN            False                   126048                   0
              True                      1542                   0
OUT           False                        0              105744
              True                         0                1275

Rows dated earlier than the row before them (same product and branch): 0


In [15]:
led = tables['stock_ledger'].copy()
led['cancelled_order'] = led['reference_id'].isin(canc_so | canc_po)
real = led[led['movement_type'] != 'ADJUSTMENT']

in_all = real[real['movement_type'] == 'IN']['quantity'].sum()
out_all = real[real['movement_type'] == 'OUT']['quantity'].sum()
in_canc = real[(real['movement_type'] == 'IN') & real['cancelled_order']]['quantity'].sum()
out_canc = real[(real['movement_type'] == 'OUT') & real['cancelled_order']]['quantity'].sum()

print('Total units IN:', in_all, '| of which on cancelled purchase orders:', in_canc)
print('Total units OUT:', out_all, '| of which on cancelled sales orders:', out_canc)
print()

def ratio(df):
    s = df.groupby(['product_id', 'movement_type'])['quantity'].sum().unstack()
    return s['IN'] / s['OUT']

kpi1_with = ratio(real)
kpi1_without = ratio(real[~real['cancelled_order']])
change = ((kpi1_without - kpi1_with) / kpi1_with * 100).abs()

print('KPI 1 average ratio, with cancelled rows:   ', round(kpi1_with.mean(), 3))
print('KPI 1 average ratio, without cancelled rows:', round(kpi1_without.mean(), 3))
print('Largest change for any single product (%):  ', round(change.max(), 2))

Total units IN: 20397795 | of which on cancelled purchase orders: 248556
Total units OUT: 1124832 | of which on cancelled sales orders: 13234

KPI 1 average ratio, with cancelled rows:    18.14
KPI 1 average ratio, without cancelled rows: 18.132
Largest change for any single product (%):   0.39


In [16]:
so = tables['sales_orders_header']
po = tables['purchase_orders_header']
ids = set(tables['stock_ledger']['reference_id'])

miss_so = so[(so['order_status'] == 'Delivered') & ~so['so_id'].isin(ids)]
miss_po = po[(po['po_status'] == 'Received') & ~po['po_id'].isin(ids)]

so_lines = tables['sales_orders_lines'].groupby('so_id').size()
po_lines = tables['purchase_orders_lines'].groupby('po_id').size()

print('SALES ORDERS delivered with no stock entry:', len(miss_so))
print(miss_so['branch_id'].value_counts().to_string())
print(pd.to_datetime(miss_so['order_date']).dt.year.value_counts().sort_index().to_string())
print('Average lines per order, missing:', round(so_lines.reindex(miss_so['so_id']).mean(), 2),
      '| all orders:', round(so_lines.mean(), 2))
print()
print('PURCHASE ORDERS received with no stock entry:', len(miss_po))
print(miss_po['branch_id'].value_counts().to_string())
print(pd.to_datetime(miss_po['order_date']).dt.year.value_counts().sort_index().to_string())
print('Average lines per order, missing:', round(po_lines.reindex(miss_po['po_id']).mean(), 2),
      '| all orders:', round(po_lines.mean(), 2))

SALES ORDERS delivered with no stock entry: 162
branch_id
CHN001    44
KOL001    36
AHM001    23
DEL001    22
HYD001    20
PUN001    17
order_date
2019    24
2020    20
2021    27
2022    24
2023    43
2024    24
Average lines per order, missing: 1.06 | all orders: 6.52

PURCHASE ORDERS received with no stock entry: 239
branch_id
DEL001    46
HYD001    44
PUN001    42
CHN001    41
AHM001    34
KOL001    32
order_date
2019    33
2020    30
2021    36
2022    53
2023    45
2024    42
Average lines per order, missing: 1.12 | all orders: 6.48


In [17]:
so_d = so[so['order_status'] == 'Delivered'].copy()
so_d['one_line'] = so_d['so_id'].map(so_lines) == 1
so_d['has_entry'] = so_d['so_id'].isin(ids)
print('DELIVERED SALES ORDERS: one-line orders vs stock entry')
print(pd.crosstab(so_d['one_line'], so_d['has_entry']))
print()
print('Sales channel of the orders with no stock entry:')
print(miss_so['sales_channel'].value_counts().to_string())
print()

po_r = po[po['po_status'] == 'Received'].copy()
po_r['one_line'] = po_r['po_id'].map(po_lines) == 1
po_r['has_entry'] = po_r['po_id'].isin(ids)
print('RECEIVED PURCHASE ORDERS: one-line orders vs stock entry')
print(pd.crosstab(po_r['one_line'], po_r['has_entry']))

DELIVERED SALES ORDERS: one-line orders vs stock entry
has_entry  False  True 
one_line               
False         10  16544
True         152   1327

Sales channel of the orders with no stock entry:
sales_channel
Dealer Network    44
Counter Sale      43
Online            43
Field Sales       32

RECEIVED PURCHASE ORDERS: one-line orders vs stock entry
has_entry  False  True 
one_line               
False         25  19745
True         214   1646


In [18]:
sl = tables['sales_orders_lines']
pl = tables['purchase_orders_lines']
led = tables['stock_ledger']

delivered = set(so.loc[so['order_status'] == 'Delivered', 'so_id'])
received = set(po.loc[po['po_status'] == 'Received', 'po_id'])

def compare(lines, id_col, order_ids, ledger_type, label):
    a = lines[lines[id_col].isin(order_ids)].groupby(id_col)['quantity'].sum()
    rows = led[(led['movement_type'] == ledger_type) & led['reference_id'].isin(order_ids)]
    b = rows.groupby('reference_id')['quantity'].sum()
    cmp = pd.DataFrame({'order_units': a, 'ledger_units': b})
    print(label)
    print('  Total units on order lines:   ', cmp['order_units'].sum())
    print('  Total units in stock ledger:  ', cmp['ledger_units'].sum())
    print('  Orders with identical units:  ', (cmp['order_units'] == cmp['ledger_units']).sum())
    print('  Orders with different units:  ', ((cmp['order_units'] != cmp['ledger_units']) & cmp['ledger_units'].notna()).sum())
    print('  Orders with no ledger entry:  ', cmp['ledger_units'].isna().sum())
    print()

compare(sl, 'so_id', delivered, 'OUT', 'DELIVERED SALES ORDERS vs ledger OUT')
compare(pl, 'po_id', received, 'IN', 'RECEIVED PURCHASE ORDERS vs ledger IN')

DELIVERED SALES ORDERS vs ledger OUT
  Total units on order lines:    1234939
  Total units in stock ledger:   1111598.0
  Orders with identical units:   9716
  Orders with different units:   8155
  Orders with no ledger entry:   162

RECEIVED PURCHASE ORDERS vs ledger IN
  Total units on order lines:    22387123
  Total units in stock ledger:   20149239.0
  Orders with identical units:   11692
  Orders with different units:   9699
  Orders with no ledger entry:   239



In [19]:
def line_level(lines, id_col, order_ids, ledger_type, label):
    a = (lines[lines[id_col].isin(order_ids)]
         .groupby([id_col, 'product_id'])['quantity'].sum().rename('order_units'))
    rows = led[(led['movement_type'] == ledger_type) & led['reference_id'].isin(order_ids)]
    b = rows.groupby(['reference_id', 'product_id'])['quantity'].sum().rename('ledger_units')
    b.index.names = [id_col, 'product_id']
    m = pd.concat([a, b], axis=1)
    print(label)
    print('  Order-product pairs:                 ', len(m))
    print('  Ledger entry with the same units:    ', (m['order_units'] == m['ledger_units']).sum())
    print('  Ledger entry with different units:   ',
          ((m['order_units'] != m['ledger_units']) & m['order_units'].notna() & m['ledger_units'].notna()).sum())
    print('  On the order but NO ledger entry:    ', m['ledger_units'].isna().sum())
    print('  In the ledger but NOT on the order:  ', m['order_units'].isna().sum())
    print()

line_level(sl, 'so_id', delivered, 'OUT', 'DELIVERED SALES ORDERS vs ledger OUT')
line_level(pl, 'po_id', received, 'IN', 'RECEIVED PURCHASE ORDERS vs ledger IN')

DELIVERED SALES ORDERS vs ledger OUT
  Order-product pairs:                  104418
  Ledger entry with the same units:     92792
  Ledger entry with different units:    2262
  On the order but NO ledger entry:     9364
  In the ledger but NOT on the order:   0

RECEIVED PURCHASE ORDERS vs ledger IN
  Order-product pairs:                  124417
  Ledger entry with the same units:     110594
  Ledger entry with different units:    2685
  On the order but NO ledger entry:     11138
  In the ledger but NOT on the order:   0



In [20]:
def why_different(lines, id_col, order_ids, ledger_type, label):
    a = (lines[lines[id_col].isin(order_ids)]
         .groupby([id_col, 'product_id'])['quantity'].agg(order_units='sum', n_lines='size'))
    rows = led[(led['movement_type'] == ledger_type) & led['reference_id'].isin(order_ids)]
    b = (rows.groupby(['reference_id', 'product_id'])['quantity']
         .agg(ledger_units='sum', n_entries='size'))
    b.index.names = [id_col, 'product_id']
    m = a.join(b)
    diff = m[m['ledger_units'].notna() & (m['order_units'] != m['ledger_units'])]
    print(label)
    print('  Pairs with an entry but different units:', len(diff))
    print('    ledger units LOWER than the order:     ', (diff['ledger_units'] < diff['order_units']).sum())
    print('    ledger units HIGHER than the order:    ', (diff['ledger_units'] > diff['order_units']).sum())
    print('    product is on 2+ lines of that order:  ', (diff['n_lines'] > 1).sum())
    print('    product is on only 1 line of the order:', (diff['n_lines'] == 1).sum())
    print()

why_different(sl, 'so_id', delivered, 'OUT', 'DELIVERED SALES ORDERS vs ledger OUT')
why_different(pl, 'po_id', received, 'IN', 'RECEIVED PURCHASE ORDERS vs ledger IN')

DELIVERED SALES ORDERS vs ledger OUT
  Pairs with an entry but different units: 2262
    ledger units LOWER than the order:      2262
    ledger units HIGHER than the order:     0
    product is on 2+ lines of that order:   2262
    product is on only 1 line of the order: 0

RECEIVED PURCHASE ORDERS vs ledger IN
  Pairs with an entry but different units: 2685
    ledger units LOWER than the order:      2685
    ledger units HIGHER than the order:     0
    product is on 2+ lines of that order:   2685
    product is on only 1 line of the order: 0



In [21]:
sh = tables['sales_orders_header']
ph = tables['purchase_orders_header']
cust = tables['customers'].set_index('customer_id')

iv = tables['invoices'].set_index('so_id')
hh = sh.set_index('so_id').loc[iv.index]
print('Invoices with a different customer than their sales order:', (iv['customer_id'] != hh['customer_id']).sum())
print('Invoices with a different branch than their sales order:   ', (iv['branch_id'] != hh['branch_id']).sum())
print()
print('Sales orders whose payment_terms differ from the customer record:', (sh['payment_terms'] != sh['customer_id'].map(cust['payment_terms'])).sum())
print('Sales orders whose branch differs from the customer record:      ', (sh['branch_id'] != sh['customer_id'].map(cust['branch_id'])).sum())
print()
out = led[led['movement_type'] == 'OUT'].merge(sh[['so_id', 'branch_id']], left_on='reference_id', right_on='so_id', suffixes=('', '_order'))
inn = led[led['movement_type'] == 'IN'].merge(ph[['po_id', 'branch_id']], left_on='reference_id', right_on='po_id', suffixes=('', '_order'))
print('Stock OUT rows logged at a different branch than their sales order:', (out['branch_id'] != out['branch_id_order']).sum())
print('Stock IN rows logged at a different branch than their purchase order:', (inn['branch_id'] != inn['branch_id_order']).sum())
print()
pr = tables['products']
odd = pr[~np.isclose(pr['margin_percentage'], (pr['unit_price'] - pr['unit_cost']) / pr['unit_cost'] * 100, atol=0.5)]
print(odd[['product_id', 'product_name', 'unit_cost', 'unit_price', 'margin_percentage']].to_string())

Invoices with a different customer than their sales order: 0
Invoices with a different branch than their sales order:    0

Sales orders whose payment_terms differ from the customer record: 0
Sales orders whose branch differs from the customer record:       0

Stock OUT rows logged at a different branch than their sales order: 0
Stock IN rows logged at a different branch than their purchase order: 0

  product_id             product_name  unit_cost  unit_price  margin_percentage
1       P002  Engine Oil Filter OF-90        450         720               37.3


In [22]:
inv = tables['invoices']
pay = tables['payments']

inv_clean = inv[~inv['invoice_id'].duplicated(keep=False)].copy()
pay_clean = pay[pay['invoice_id'].isin(inv_clean['invoice_id'])]

paid_sum = pay_clean.groupby('invoice_id')['payment_amount'].sum()
inv_clean['paid_sum'] = inv_clean['invoice_id'].map(paid_sum).fillna(0)
inv_clean['share_paid'] = inv_clean['paid_sum'] / inv_clean['grand_total']

def bucket(x):
    if x == 0:
        return 'nothing paid'
    if x < 0.999:
        return 'part paid'
    if x <= 1.001:
        return 'fully paid'
    return 'overpaid'

inv_clean['actual'] = inv_clean['share_paid'].apply(bucket)
print('Invoices checked:', len(inv_clean))
print()
print(pd.crosstab(inv_clean['payment_status'], inv_clean['actual']))

Invoices checked: 17640

actual          fully paid  nothing paid  part paid
payment_status                                     
Paid                 12264             0          0
Partially Paid           0             0       3578
Unpaid                   0          1798          0


In [23]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

tables = ['branches', 'customers', 'inventory_master', 'invoices',
          'payments', 'products', 'purchase_orders_header',
          'purchase_orders_lines', 'sales_orders_header',
          'sales_orders_lines', 'stock_ledger', 'suppliers']

for t in tables:
    df = pd.read_csv(BASE_PATH + t + '.csv', nrows=5)
    print(t, '->', list(df.columns))
    print()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
branches -> ['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']

customers -> ['customer_id', 'customer_type', 'industry_segment', 'city', 'state', 'pincode', 'region', 'branch_id', 'credit_limit', 'current_balance', 'payment_terms', 'customer_since', 'last_purchase_date', 'total_purchase_value', 'customer_rating']

inventory_master -> ['product_id', 'branch_id', 'opening_stock', 'reorder_level', 'safety_stock', 'max_stock', 'current_stock', 'warehouse_bin']

invoices -> ['invoice_id', 'so_id', 'customer_id', 'branch_id', 'invoice_date', 'due_date', 'total_order_value', 'total_gst_amount', 'grand_total', 'payment_status']

payments -> ['payment_id', 'invoice_id', 'payment_date', 'pay

In [24]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

names = ['branches', 'customers', 'inventory_master', 'invoices',
         'payments', 'products', 'purchase_orders_header',
         'purchase_orders_lines', 'sales_orders_header',
         'sales_orders_lines', 'stock_ledger', 'suppliers']
t = {n: pd.read_csv(BASE_PATH + n + '.csv') for n in names}

# (child table, child column, parent table, parent column)
links = [
    ('customers', 'branch_id', 'branches', 'branch_id'),
    ('inventory_master', 'product_id', 'products', 'product_id'),
    ('inventory_master', 'branch_id', 'branches', 'branch_id'),
    ('invoices', 'so_id', 'sales_orders_header', 'so_id'),
    ('invoices', 'customer_id', 'customers', 'customer_id'),
    ('invoices', 'branch_id', 'branches', 'branch_id'),
    ('payments', 'invoice_id', 'invoices', 'invoice_id'),
    ('purchase_orders_header', 'supplier_id', 'suppliers', 'supplier_id'),
    ('purchase_orders_header', 'branch_id', 'branches', 'branch_id'),
    ('purchase_orders_lines', 'po_id', 'purchase_orders_header', 'po_id'),
    ('purchase_orders_lines', 'product_id', 'products', 'product_id'),
    ('sales_orders_header', 'customer_id', 'customers', 'customer_id'),
    ('sales_orders_header', 'branch_id', 'branches', 'branch_id'),
    ('sales_orders_lines', 'so_id', 'sales_orders_header', 'so_id'),
    ('sales_orders_lines', 'product_id', 'products', 'product_id'),
    ('stock_ledger', 'product_id', 'products', 'product_id'),
    ('stock_ledger', 'branch_id', 'branches', 'branch_id'),
]

print('PART 1: orphan check (child values with no match in the parent)')
for ct, cc, pt, pc in links:
    parent_ids = set(t[pt][pc].dropna())
    child = t[ct][cc]
    empty = child.isna().sum()
    orphans = (child.notna() & ~child.isin(parent_ids)).sum()
    print(f'{ct}.{cc} -> {pt}.{pc}: rows {len(child)}, empty {empty}, orphans {orphans}')

print()
print('PART 2: stock_ledger.reference_id, split by reference_type')
led = t['stock_ledger']
po_ids = set(t['purchase_orders_header']['po_id'])
so_ids = set(t['sales_orders_header']['so_id'])
for rtype, ids in [('PO', po_ids), ('SO', so_ids)]:
    rows = led[led['reference_type'] == rtype]
    print(f'{rtype}: rows {len(rows)}, orphans {(~rows["reference_id"].isin(ids)).sum()}')
adj = led[led['reference_type'] == 'ADJ']
print(f'ADJ: rows {len(adj)}, matching a PO or SO: {adj["reference_id"].isin(po_ids | so_ids).sum()}')

print()
print('PART 3: one-to-one and unique-key claims')
inv = t['invoices']
soh = t['sales_orders_header']
delivered = set(soh.loc[soh['order_status'] == 'Delivered', 'so_id'])
inv_so = set(inv['so_id'])
print('invoices with a repeated so_id:', inv['so_id'].duplicated().sum())
print('Delivered orders with no invoice:', len(delivered - inv_so))
print('invoices whose order is not Delivered:', len(inv_so - delivered))
print('inventory_master repeated (product_id, branch_id):',
      t['inventory_master'].duplicated(['product_id', 'branch_id']).sum())
print('sales_orders_lines repeated (so_id, line_number):',
      t['sales_orders_lines'].duplicated(['so_id', 'line_number']).sum())
print('purchase_orders_lines repeated (po_id, line_number):',
      t['purchase_orders_lines'].duplicated(['po_id', 'line_number']).sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PART 1: orphan check (child values with no match in the parent)
customers.branch_id -> branches.branch_id: rows 500, empty 0, orphans 0
inventory_master.product_id -> products.product_id: rows 180, empty 0, orphans 0
inventory_master.branch_id -> branches.branch_id: rows 180, empty 0, orphans 0
invoices.so_id -> sales_orders_header.so_id: rows 18033, empty 0, orphans 0
invoices.customer_id -> customers.customer_id: rows 18033, empty 0, orphans 0
invoices.branch_id -> branches.branch_id: rows 18033, empty 0, orphans 0
payments.invoice_id -> invoices.invoice_id: rows 19257, empty 0, orphans 0
purchase_orders_header.supplier_id -> suppliers.supplier_id: rows 24000, empty 0, orphans 0
purchase_orders_header.branch_id -> branches.branch_id: rows 24000, empty 0, orphans 0
purchase_orders_lines.po_id -> purchase_orders_header.po_id: rows 155495, empty 0, orphans 0
p